In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.DataFrame({
    "income": [25000, 28000, 70000, 75000, 150000, 160000],
    "spending": [10000, 12000, 40000, 43000, 85000, 90000]
})

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[["income", "spending"]])

In [4]:
model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["cluster"] = model.fit_predict(X_scaled)
print(df)

   income  spending  cluster
0   25000     10000        2
1   28000     12000        2
2   70000     40000        0
3   75000     43000        0
4  150000     85000        1
5  160000     90000        1


In [5]:
from sklearn.metrics import silhouette_score

results = []

for k in range(2, 6):
    model_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model_k.fit_predict(X_scaled)

    results.append({
        "K": k,
        "inertia": model_k.inertia_,
        "silhouette": silhouette_score(X_scaled, labels)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3))

   K  inertia  silhouette
0  2    1.726       0.741
1  3    0.043       0.905
2  4    0.013       0.610
3  5    0.004       0.310


In [6]:
final_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["cluster"] = final_model.fit_predict(X_scaled)

summary = df.groupby("cluster").agg(
    customers=("income", "size"),
    avg_income=("income", "mean"),
    avg_spending=("spending", "mean")
)

print(summary.round(2))

         customers  avg_income  avg_spending
cluster                                     
0                2     72500.0       41500.0
1                2    155000.0       87500.0
2                2     26500.0       11000.0
